# 第10讲：数字人文 MLLM 研究管线——毕业设计原型
## 「视频-文献-分析」端到端数字人文研究原型

本讲是全课的**收官讲**。前九讲分别构建了语音转写、文本摘要、领域微调、跨模态对齐、视觉分析、评估量化、可视化呈现等独立组件——本讲将它们串联为一条完整的研究管线：输入一段数字人文短视频，输出一份可直接用于论文的结构化研究报告。

> **数据来源声明**：本讲使用的数字人文训练语料为人工合成的10条示例（甲骨文、古籍修复等），经人工核验，仅用于教学演示，不代表任何特定档案馆或研究机构的真实数据。

# 第0章  课程回顾：九讲的旅程

在进入管线之前，我们先梳理前九讲各自构建了什么，以及它们在本讲中扮演的角色。

## 第1讲 — LLaMA-3 文本摘要

**构建了什么**：加载 `NousResearch/Meta-Llama-3-8B-Instruct`，对视频转写文本生成2-3句中文摘要。

**核心函数**：`summarize_transcript(text)` → 返回字符串摘要

**在本讲的作用**：管线第二阶段——将 Whisper 转写文本压缩为可检索的摘要条目。

## 第2讲 — CLIP 跨模态对齐

**构建了什么**：用 CLIP 将视频帧与古籍文本映射到同一嵌入空间，计算相似度矩阵。

**核心概念**：图像与文本的嵌入对齐——输出相似度分数，不生成文字。

**在本讲的作用**：概念基础——第7讲的 LLaVA-Next 是其生成式升级版，本讲直接使用 LLaVA-Next。

## 第3讲 — QLoRA 微调：数字人文术语适配

**构建了什么**：用 QLoRA 对 LLaMA-3 做领域微调，使其能准确解释「刻本」「金石学」等 DH 专业术语。

**核心产物**：LoRA 适配器权重，保存在 `./qlora_adapter_lecture3`（约 55 MB）

**在本讲的作用**：管线第二阶段——对转写文本生成术语解释，弥补通用模型的领域盲区。

## 第4讲 — 适配器推理与领域对比

**构建了什么**：加载第3讲适配器，对比微调前后的术语解释质量，验证 LoRA 的领域迁移效果。

**核心结论**：微调后术语准确率提升，但训练数据规模决定了上限。

**在本讲的作用**：验证基础——本讲复用相同的适配器加载逻辑。

## 第5讲 — BLIP-2 视频帧解说

**构建了什么**：用 BLIP-2 对视频帧生成英文描述，提取均匀采样帧，拼合为结构化归档记录。

**核心产物**：逐帧英文解说 JSON，帧提取工具函数。

**在本讲的作用**：帧提取逻辑的前身——本讲使用 OpenCV 提取中间帧，送入 LLaVA-Next。

## 第6讲 — Whisper 语音转写

**构建了什么**：用 Whisper 对视频音频做自动语音识别（ASR），生成带时间戳的中文转写文本。

**核心函数**：`transcribe_audio(video_path)` → 返回转写字符串

**在本讲的作用**：管线第一阶段——从视频中提取文本内容，作为后续所有文本分析的输入。

## 第7讲 — LLaVA-Next 多模态视觉分析

**构建了什么**：加载 `llava-hf/llava-v1.6-mistral-7b-hf`（4位量化），对数字人文影像做三种分析：文字识别、文化元素识别、标准化档案描述。

**核心函数**：`analyze_dh_frame(image, tasks)` → 返回分析结果字典

**在本讲的作用**：管线第三阶段——对提取的视频帧做视觉内容分析。

## 第8讲 — BLEU 与术语准确率评估

**构建了什么**：用 `sacrebleu` 计算 BLEU 分数，用关键词匹配计算术语准确率，对微调模型的术语解释质量做量化评估。

**核心函数**：`compute_bleu(references, hypotheses)` 和 `compute_term_accuracy(term_set, output)`

**在本讲的作用**：管线第四阶段——对管线各模块的输出质量做数值化评估。

## 第9讲 — 评估结果可视化

**构建了什么**：用 matplotlib 将 BLEU 和术语准确率绘制为论文级图表（折线图 + 柱状图），移除视觉噪声，统一字号配色。

**核心函数**：`plot_evaluation_results(metrics_dict)` → 渲染 matplotlib 图表

**在本讲的作用**：管线第四阶段输出——将评估数字转化为论文可用的可视化结果。

# 第1章  环境准备与 GPU 自检

本讲同时加载 LLaMA-3（含 LoRA 适配器）和 LLaVA-Next（4位量化），合计约 **14 GB 显存**。

- T4（16 GB）：刚好可以运行，请确保没有其他进程占用显存
- 低于 12 GB：将发出警告，部分组件可能 OOM

> **加载顺序**：先加载文本模型（LLaMA-3 + LoRA），完成文本分析后释放，再加载视觉模型（LLaVA-Next）——这样可以避免两个模型同时占用显存。

In [ ]:
# ===== 第1章：环境准备 =====
# Colab 用户：运行此单元格自动安装依赖。
# 本地 mllm-dh 环境用户：可注释掉 !pip 行。

!pip install -q transformers peft accelerate bitsandbytes openai-whisper sacrebleu matplotlib Pillow opencv-python

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        '未检测到 GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → T4 GPU。'
        ' 本讲同时加载两个大模型，必须在 GPU 上运行。'
    )

device_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU 已就绪：{device_name}')
print(f'显存总量：{total_mem:.1f} GB')

if total_mem < 12.0:
    print()
    print('WARNING: 显存低于 12 GB，加载全部组件可能触发 OOM。')
    print('建议使用 T4（16 GB）或更高规格的 GPU。')
else:
    print('显存充足，可以加载全部组件。')

# 第2章  加载全部组件

依次加载五个组件，每个单元格只做一件事：

| 组件 | 来源 | 显存占用 |
|------|------|----------|
| LLaMA-3 基础模型 | NousResearch/Meta-Llama-3-8B-Instruct | ~5 GB（4位量化） |
| L3 LoRA 适配器 | ./qlora_adapter_lecture3 | +55 MB |
| LLaVA-Next | llava-hf/llava-v1.6-mistral-7b-hf | ~5 GB（4位量化）|
| Whisper | openai/whisper-tiny | ~150 MB |
| 评估+可视化函数 | 内联定义（来自 L8/L9） | 无模型 |

In [ ]:
# ===== 2a：加载 LLaMA-3 基础模型（4位量化）=====
# 来自第1/3/4讲——NousResearch 镜像，无需登录

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL_ID = 'NousResearch/Meta-Llama-3-8B-Instruct'

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model.eval()
print(f'LLaMA-3 基础模型加载完成（{sum(p.numel() for p in base_model.parameters())/1e9:.2f} B 参数）')

In [ ]:
# ===== 2b：加载 L3 LoRA 适配器 =====
# 来自第3讲——如适配器不存在，硬性报错并给出明确指引

import os
from peft import PeftModel

ADAPTER_PATH = './qlora_adapter_lecture3'

if not os.path.exists(ADAPTER_PATH):
    raise FileNotFoundError(
        f'未找到 L3 LoRA 适配器：{ADAPTER_PATH}\n'
        '请执行以下步骤之一：\n'
        '  (A) 先运行第3讲 notebook，在最后一步保存并下载适配器 zip，\n'
        '      再上传并解压到本 notebook 的当前目录。\n'
        '  (B) 从课程仓库 shared/weights/qlora_adapter_lecture3_archive/ 下载，\n'
        '      解压后重命名为 qlora_adapter_lecture3/。'
    )

text_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
text_model.config.use_cache = True
text_model.eval()
print(f'L3 LoRA 适配器加载完成：{ADAPTER_PATH}')

In [ ]:
# ===== 2c：加载 LLaVA-Next（4位量化）=====
# 来自第7讲——使用专用处理器和模型类，非 AutoModel
# 首次运行需下载约 4 GB 权重，Colab 约需 3-5 分钟

from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration

LLAVA_MODEL_ID = 'llava-hf/llava-v1.6-mistral-7b-hf'

llava_processor = LlavaNextProcessor.from_pretrained(LLAVA_MODEL_ID)
print('LLaVA-Next 处理器加载完成')

llava_model = LlavaNextForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
llava_model.eval()
print('LLaVA-Next 模型加载完成')
print(f'显存占用（加载后）：{torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# ===== 2d：加载 Whisper（tiny，速度优先）=====
# 来自第6讲——whisper-tiny 约 150 MB，在 T4 上每分钟音频约需 10 秒
# 如需更高转写精度，可将 'tiny' 替换为 'base' 或 'small'

import whisper

whisper_model = whisper.load_model('tiny')
print(f'Whisper-tiny 加载完成，设备：{next(whisper_model.parameters()).device}')

In [ ]:
# ===== 2e：内联定义评估与可视化函数（来自 L8/L9）=====
# 不从其他 notebook 导入——在此直接复现，确保本讲可独立运行
# 每个函数不超过10行，与 L8/L9 原始签名一致

from sacrebleu.metrics import BLEU
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# ---- 来自 L8：BLEU（中文字符级分词，返回 0–1 浮点数）----
def compute_bleu(references: list, hypotheses: list) -> float:
    """
    计算语料级 BLEU（sacrebleu，zh 中文字符级分词）。
    参数：references / hypotheses 为等长字符串列表。
    返回：float，范围 0–1（sacrebleu 原始分数除以100）。
    """
    bleu_metric = BLEU(tokenize='zh', effective_order=True)
    score = bleu_metric.corpus_score(hypotheses, [references])
    return round(score.score / 100.0, 4)   # 归一化为 0–1

# ---- 来自 L8：术语准确率（子串匹配，返回 0–1 浮点数）----
# L3 训练数据覆盖的10个核心术语 + L8/L9 评估中补充的7个
DH_TERM_SET = {
    '甲骨文', '古籍修复', '非物质文化遗产', '敦煌壁画', '活字印刷',
    '青铜器铭文', '昆曲', '简牍', '拓片', '二十四节气',
    '刻本', '缮写', '金石学', '校勘', '传世本', '目录学', '数字孪生',
}

def compute_term_accuracy(term_references: list, outputs: list) -> float:
    """
    计算术语准确率：检查 outputs 中出现了多少 DH_TERM_SET 里的术语。
    参数：term_references（参考文本列表，用于确认哪些术语应出现）；
          outputs（模型生成的文本列表）。
    返回：float，范围 0–1（命中术语数 / 术语集合大小）。
    """
    combined_output = ' '.join(outputs)
    hits = sum(1 for t in DH_TERM_SET if t in combined_output)
    return round(hits / len(DH_TERM_SET), 4) if DH_TERM_SET else 0.0

# ---- 来自 L9：论文级评估结果可视化（接受 0–1 量纲的指标值）----
def plot_evaluation_results(metrics_dict: dict):
    """
    将评估指标绘制为论文级柱状图（来自第9讲）。
    参数：metrics_dict，键为指标名，值为 0–1 范围的浮点数。
    """
    labels = list(metrics_dict.keys())
    values = [float(v) for v in metrics_dict.values()]

    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ['#2B579A', '#D64531', '#4CAF50', '#FF9800'][:len(labels)]
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.5)

    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0, min(max(values) * 1.35, 1.0) if values else 1.0)
    ax.set_ylabel('分数（0–1）')
    ax.set_title('管线评估结果', fontweight='bold')
    plt.tight_layout()
    plt.show()

print('评估与可视化函数定义完成（L8/L9 内联版）')
print(f'  compute_bleu:          返回 0–1 浮点数（sacrebleu / 100）')
print(f'  compute_term_accuracy: 返回 0–1 浮点数（命中/总术语数）')
print(f'  plot_evaluation_results: 接受 0–1 量纲 dict')

In [ ]:
# ===== 检查点：确认所有组件加载成功 =====

components = {
    'LLaMA-3 基础模型（L1/L3/L4）': text_model is not None,
    'L3 LoRA 适配器（L3）': os.path.exists(ADAPTER_PATH),
    'LLaVA-Next 模型（L7）': llava_model is not None,
    'LLaVA-Next 处理器（L7）': llava_processor is not None,
    'Whisper-tiny（L6）': whisper_model is not None,
    '评估函数 compute_bleu（L8）': callable(compute_bleu),
    '评估函数 compute_term_accuracy（L8）': callable(compute_term_accuracy),
    '可视化函数 plot_evaluation_results（L9）': callable(plot_evaluation_results),
}

print('===== 组件加载状态 =====')
all_ok = True
for name, status in components.items():
    icon = 'OK' if status else 'FAIL'
    print(f'  [{icon}] {name}')
    if not status:
        all_ok = False

print()
if all_ok:
    print('所有组件就绪，可以进入管线。')
else:
    raise RuntimeError('部分组件未加载成功，请检查上方报错信息后重新运行。')

# 第3章  输入：数字人文视频

## 这是管线的唯一入口——一段视频进入，一份完整研究报告输出

本章是整个管线的**单一入口**：上传一段数字人文短视频，系统将自动完成所有后续阶段：

```
视频文件 (mp4/mov/mkv)
  ├── → 提取中间帧 (OpenCV)  →  第5章 LLaVA-Next 视觉分析
  └── → 提取音频，Whisper 转写  →  第4章 LLaMA-3 文本分析
```

适合本讲的视频类型：
- 文化遗产纪录片片段（含人物讲解）
- 古籍、文物介绍短视频
- 博物馆展厅导览视频

> **时长建议**：30 秒至 3 分钟。过长的视频会增加 Whisper 转写时间（whisper-tiny：约 1 分钟音频 ≈ 10 秒处理时间）。
>
> **没有视频？** 请前往 CC 授权的视频库（如 Vimeo 或 YouTube）下载一段数字文化遗产相关视频用于测试。管线不会静默降级——如果文件不存在，会立即报错并告诉你该做什么。

In [ ]:
# ===== 第3章：上传视频（Colab / 本地双路支持）=====

import glob
from IPython.display import display as ipy_display
from PIL import Image

# ---- 本地用户：将此处改为你的视频文件路径 ----
LOCAL_VIDEO_PATH = 'dh_video.mp4'

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

video_path = None

if IN_COLAB:
    print('请在弹出的文件选择器中上传一段数字人文短视频...')
    uploaded = files.upload()
    if uploaded:
        video_path = list(uploaded.keys())[0]
        print(f'已上传：{video_path}')
    else:
        raise FileNotFoundError(
            '未选择视频文件。请重新运行此单元格并上传视频。'
            ' 本讲管线需要真实视频作为输入——不提供占位替代。'
        )
elif os.path.exists(LOCAL_VIDEO_PATH):
    video_path = LOCAL_VIDEO_PATH
    print(f'已找到本地视频：{video_path}')
else:
    candidates = glob.glob('*.mp4') + glob.glob('*.mov') + glob.glob('*.mkv')
    if candidates:
        video_path = candidates[0]
        print(f'自动检测到视频文件：{video_path}')
    else:
        raise FileNotFoundError(
            f'未找到视频文件（尝试路径：{LOCAL_VIDEO_PATH}）。'
            ' 请将视频文件放在 notebook 同目录，或将 LOCAL_VIDEO_PATH 改为正确路径。'
        )

print(f'视频路径确认：{video_path}')

In [ ]:
# ===== 第3章（续）：提取视频中间帧 =====
# 选取中间帧作为视觉分析的代表帧——比第1帧更有代表性

import cv2

cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

if total_frames <= 0:
    cap.release()
    raise RuntimeError(
        f'视频文件可能损坏或格式不受支持（总帧数={total_frames}）。'
        ' 请尝试转换为 mp4（H.264 编码）后重新上传。'
    )

mid_frame_idx = total_frames // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame_idx)
ret, frame_bgr = cap.read()
cap.release()

if not ret:
    raise RuntimeError('无法读取视频中间帧，请检查视频文件完整性。')

# BGR → RGB → PIL Image（LLaVA-Next 使用 PIL 格式）
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
dh_frame = Image.fromarray(frame_rgb)

duration_sec = total_frames / fps if fps > 0 else 0
print(f'视频信息：总帧数 {total_frames}，帧率 {fps:.1f} fps，时长约 {duration_sec:.1f} 秒')
print(f'已提取中间帧（第 {mid_frame_idx} 帧），尺寸：{dh_frame.size}')
print('预览帧图像：')
ipy_display(dh_frame.resize((480, int(480 * dh_frame.height / dh_frame.width))))

In [ ]:
# ===== 第3章（续）：Whisper 语音转写 =====
# 来自第6讲——将视频音频转写为中文文本
# whisper-tiny 每分钟音频约需 10 秒，请耐心等待

print('正在转写音频...')
whisper_result = whisper_model.transcribe(video_path, language='zh', fp16=torch.cuda.is_available())
transcript = whisper_result['text'].strip()

if not transcript:
    raise RuntimeError(
        'Whisper 未能转写出任何文字。可能原因：\n'
        '  (1) 视频无人声或音量过低\n'
        '  (2) 语言不是中文（尝试将 language 参数改为 "auto"）\n'
        '  (3) 视频格式不含音频轨道'
    )

print()
print('===== Whisper 转写结果 =====')
print(transcript)
print('=' * 35)

# 第4章  第二阶段：文本分析

## 用 L1 摘要 + L3 术语解释，双路处理转写文本

| 子任务 | 模型 | 来源讲 | 输出 |
|--------|------|--------|------|
| 摘要生成 | LLaMA-3（通用推理） | L1 | 2-3句中文摘要 |
| 术语解释 | LLaMA-3 + LoRA 适配器 | L3 | 数字人文专业术语解释 |

两个任务使用同一个模型实例（`text_model`），只是系统提示词不同。

In [ ]:
# ===== 第4章：定义文本生成辅助函数 =====
# 封装 LLaMA-3 的推理流程，供摘要和术语解释共用

def generate_text(prompt_messages: list, max_new_tokens: int = 256) -> str:
    """向 text_model 发送对话格式提示词，返回生成文本。"""
    formatted = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(text_model.device)

    with torch.no_grad():
        output_ids = text_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# 来自 L1：摘要生成
def summarize_transcript(text: str) -> str:
    """对视频转写文本生成2-3句中文摘要（来自第1讲）。"""
    messages = [
        {'role': 'system', 'content': '你是一位数字人文研究助手，擅长对视频内容做简洁的学术摘要。'},
        {'role': 'user', 'content': f'请用2-3句话总结以下视频转写内容：\n\n{text}'},
    ]
    return generate_text(messages, max_new_tokens=150)


# 来自 L3：术语解释（使用 LoRA 适配器赋予的领域知识）
def explain_dh_terminology(text: str) -> str:
    """识别并解释转写文本中的数字人文专业术语（来自第3讲 LoRA 适配器）。"""
    messages = [
        {'role': 'system', 'content': '你是一位专精于数字人文领域的学者，能够准确解释古籍文献、文化遗产与档案实践中的专业术语。'},
        {'role': 'user', 'content': text},
    ]
    return generate_text(messages, max_new_tokens=250)


print('文本生成辅助函数定义完成（summarize_transcript / explain_dh_terminology）')

In [ ]:
# ===== 4a：运行摘要生成（来自 L1）=====

print('正在生成摘要...')
summary = summarize_transcript(transcript)

print()
print('===== L1 摘要输出 =====')
print(summary)
print('=' * 35)

In [ ]:
# ===== 4b：运行术语解释（来自 L3 LoRA 适配器）=====

print('正在生成术语解释（使用 L3 LoRA 适配器）...')
terminology_explanation = explain_dh_terminology(transcript)

print()
print('===== L3 术语解释输出 =====')
print(terminology_explanation)
print('=' * 35)

In [ ]:
# ===== 4c：存储文本分析结果 =====

results = {
    'summary': summary,
    'terminology': terminology_explanation,
}

print('文本分析阶段完成，结果已存储：')
for k, v in results.items():
    print(f'  {k}: {v[:60]}...')

# 第5章  第三阶段：视觉分析

## 用 LLaVA-Next 对视频帧做三种分析（来自 L7）

输入：第3章提取的视频中间帧（PIL Image）

| 分析任务 | 提示词目标 | DH 应用场景 |
|----------|-----------|------------|
| `text` | 识别并转录可见文字 | 古籍字符提取、碑刻转录 |
| `cultural` | 描述文物材质/形制/纹饰 | 文物档案著录 |
| `archival` | 生成标准化四字段档案描述 | 元数据入库 |

In [ ]:
# ===== 第5章：定义 LLaVA-Next 推理函数（来自 L7）=====

def run_llava(image: Image.Image, prompt: str, max_new_tokens: int = 300) -> str:
    """向 LLaVA-Next 发送图像+提示词，返回生成文本（来自第7讲）。"""
    conversation = [{
        'role': 'user',
        'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}],
    }]
    prompt_text = llava_processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = llava_processor(images=image, text=prompt_text, return_tensors='pt').to(llava_model.device)

    with torch.no_grad():
        output_ids = llava_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return llava_processor.decode(new_tokens, skip_special_tokens=True).strip()


def analyze_dh_frame(image: Image.Image, tasks: list = None) -> dict:
    """对单张数字人文影像帧运行指定分析任务（来自第7讲）。"""
    if tasks is None:
        tasks = ['text', 'cultural', 'archival']

    prompts = {
        'text': (
            '请仔细观察图像，识别并转录所有可见的文字或字符。'
            '如果文字模糊，请尽力辨认并用方括号标注不确定的部分。'
            '按照从上到下、从左到右的顺序转录。'
        ),
        'cultural': (
            '请描述图像中可见的文化元素、文物或历史物品，包括：'
            '（1）材质；（2）形制；（3）纹饰。如有不确定之处请说明。'
        ),
        'archival': (
            '请为这张图像生成标准化档案描述，包含以下四部分：\n'
            '1.【图像内容概述】2.【历史与文化元素】3.【图像质量评估】4.【建议档案分类标签】'
        ),
    }
    max_tokens = {'text': 300, 'cultural': 300, 'archival': 400}

    return {
        task: run_llava(image, prompts[task], max_new_tokens=max_tokens[task])
        for task in tasks if task in prompts
    }


print('LLaVA-Next 分析函数定义完成（run_llava / analyze_dh_frame）')

In [ ]:
# ===== 第5章：对视频中间帧运行三项视觉分析 =====
# 每项分析约需 15-30 秒，共约 1-2 分钟

print('正在运行视觉分析（三项任务）...')
visual_results = analyze_dh_frame(dh_frame, tasks=['text', 'cultural', 'archival'])

print()
print('===== 任务一：文字识别 =====')
print(visual_results.get('text', '（未运行）'))

print()
print('===== 任务二：文化元素识别 =====')
print(visual_results.get('cultural', '（未运行）'))

print()
print('===== 任务三：标准化档案描述 =====')
print(visual_results.get('archival', '（未运行）'))
print('=' * 35)

In [ ]:
# ===== 第5章：存储视觉分析结果 =====

results['visual_text'] = visual_results.get('text', '')
results['cultural_elements'] = visual_results.get('cultural', '')
results['archival_desc'] = visual_results.get('archival', '')

print('视觉分析阶段完成，结果已存入 results 字典。')
print(f'当前 results 键：{list(results.keys())}')

# 第6章  第四阶段：评估

## 用 L8 的两个指标对管线输出做量化评估

| 指标 | 计算对象 | 来源讲 | 返回值 |
|------|---------|-------|-------|
| BLEU | 摘要 vs 原始转写（代理参考） | L8 | 0–1 浮点数 |
| 术语准确率 | 术语解释中命中已知 DH 术语的比例 | L8 | 0–1 浮点数 |

> **代理评估的局限**：在单条样本上计算 BLEU 是**代理评估**——用转写原文当"参考答案"并不标准，因为好的摘要不应该与原文高度重叠。这里的目的是展示管线端到端可运行，为研究者提供即时质量信号。
>
> **正式评估**需要：(1) 至少 50 条人工标注的参考摘要，(2) 独立测试集（不含训练数据），(3) 与基线模型对比。参见 L8 的完整方法论说明。

In [ ]:
# ===== 第6章：计算评估指标（来自 L8）=====
# 两个函数均返回 0–1 浮点数，便于在同一量纲下比较

# BLEU：摘要 vs 转写原文（单条，代理参考；0–1 归一化）
bleu_score = compute_bleu(
    references=[transcript],
    hypotheses=[summary],
)

# 术语准确率：检查术语解释中命中了多少已知 DH 术语（0–1 归一化）
term_acc = compute_term_accuracy(
    term_references=[transcript],           # 参考文本（转写原文）
    outputs=[terminology_explanation],      # 待评估输出（L3 术语解释）
)

# 两个指标均为 0–1，可直接传入 plot_evaluation_results
metrics = {
    'BLEU（摘要）': bleu_score,
    '术语命中率': term_acc,
}

print('===== 评估结果 =====')
print(f'  BLEU（摘要 vs 转写）：{bleu_score:.4f}（即 {bleu_score*100:.2f} 分）')
print(f'  术语命中率：{term_acc:.4f}（即 {term_acc*100:.1f}%，命中 {int(term_acc*len(DH_TERM_SET))}/{len(DH_TERM_SET)} 个已知术语）')
print()
print('注：以上为单条样本代理评估，正式评估需 50+ 条人工标注样本（参见 L8）。')

# 第7章  可视化评估结果（来自 L9）

将评估数字转化为论文可用的图表——与 L9 的 `plot_evaluation_results()` 函数接口完全一致。

In [ ]:
# ===== 第7章：绘制评估结果图表（来自 L9）=====

print('===== 评估结果可视化 =====')
plot_evaluation_results(metrics)

In [ ]:
# ===== 第7章（续）：研究报告快照表格 =====
# 用 matplotlib 表格呈现全部管线输出，适合在论文附录或展示 slides 中使用

def truncate(s, n=55):
    s = str(s).replace('\n', ' ')
    return s[:n] + '...' if len(s) > n else s

table_data = [
    ['转写文本（Whisper，L6）', truncate(transcript)],
    ['摘要（LLaMA-3，L1）', truncate(summary)],
    ['术语解释（QLoRA，L3）', truncate(terminology_explanation)],
    ['视觉-文字识别（LLaVA-Next，L7）', truncate(results.get('visual_text', ''))],
    ['视觉-文化元素（LLaVA-Next，L7）', truncate(results.get('cultural_elements', ''))],
    ['档案描述（LLaVA-Next，L7）', truncate(results.get('archival_desc', ''))],
    ['BLEU 分数（0–1 归一化，L8）', f'{bleu_score:.4f}'],
    ['术语命中率（0–1 归一化，L8）', f'{term_acc:.4f}（{term_acc*100:.1f}%）'],
]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis('off')
tbl = ax.table(
    cellText=table_data,
    colLabels=['字段', '内容（截断）'],
    cellLoc='left',
    loc='center',
    colWidths=[0.28, 0.72],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.6)

# 表头加深色背景
for col in range(2):
    tbl[(0, col)].set_facecolor('#2B579A')
    tbl[(0, col)].set_text_props(color='white', fontweight='bold')

ax.set_title('研究报告快照', fontweight='bold', fontsize=12, pad=10)
plt.tight_layout()
plt.show()
print('快照表格已渲染。')

# 第8章  结构化输出：研究报告

## 管线的最终交付物

将全部管线输出整合为一个标准 Python 字典 `research_report`，并保存为 JSON 文件。

这是研究者**可以直接存档、检索、二次分析**的格式——不是中间产物，而是研究报告本身：

```python
research_report = {
    "input_video":              # 原始视频路径
    "transcript":               # Whisper 转写全文（L6）
    "summary":                  # LLaMA-3 摘要（L1）
    "terminology_explanation":  # QLoRA 术语解释（L3）
    "visual_analysis": {...},   # LLaVA-Next 三项分析（L7）
    "evaluation": {             # 量化评估（L8）
        "bleu": float,          #   0–1 归一化 BLEU
        "term_accuracy": float, #   0–1 归一化术语命中率
    },
    "generated_at":             # 时间戳，用于档案溯源
}
```

> 保存为 JSON 后可直接用 pandas、ElasticSearch 或任意数据库进行检索与分析。

In [ ]:
# ===== 第8章：组装结构化研究报告 =====

import json
from datetime import datetime

research_report = {
    'input_video': video_path,
    'transcript': transcript,
    'summary': summary,
    'terminology_explanation': terminology_explanation,
    'visual_analysis': {
        'text_extraction': results.get('visual_text', ''),
        'cultural_elements': results.get('cultural_elements', ''),
        'archival_description': results.get('archival_desc', ''),
    },
    'evaluation': {
        'bleu': float(bleu_score),           # 0–1 归一化（sacrebleu / 100）
        'term_accuracy': float(term_acc),    # 0–1 归一化（命中/术语集大小）
        'note': '单条样本代理评估，正式评估需 50+ 条人工标注参考答案（参见 L8）',
    },
    'pipeline_components': [
        'Whisper-tiny（L6）',
        'LLaMA-3 8B + QLoRA 适配器（L1/L3）',
        'LLaVA-Next Mistral-7B 4bit（L7）',
        'sacrebleu BLEU + 术语匹配（L8）',
        'matplotlib 论文图表（L9）',
    ],
    'generated_at': datetime.now().isoformat(),
}

# 保存为 JSON 文件
OUTPUT_PATH = 'L10_research_report.json'
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(research_report, f, ensure_ascii=False, indent=2)

print(f'研究报告已保存至：{OUTPUT_PATH}')
print(f'文件大小：{os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB')
print()
print('===== 研究报告（JSON 预览）=====')
# 打印时截断长文本字段
preview = {k: (v[:100] + '...' if isinstance(v, str) and len(v) > 100 else v)
           for k, v in research_report.items()}
print(json.dumps(preview, ensure_ascii=False, indent=2))

In [ ]:
# ===== 第8章（续）：Colab 用户下载研究报告 =====
# 运行时重置后文件会丢失——请立即下载

if IN_COLAB:
    files.download(OUTPUT_PATH)
    print(f'已触发下载：{OUTPUT_PATH}')
    print('如果下载未自动开始，请在左侧文件浏览器中右键点击该文件并选择「下载」。')
else:
    print(f'本地用户：报告已保存至 {os.path.abspath(OUTPUT_PATH)}')

# 第9章  创新方向

## 方向一：扩充语料规模

**当前瓶颈**：L3 的 QLoRA 微调仅使用10条合成示例，术语覆盖率有限。

**改进路径**：
- 引入真实 DH 语料，例如 **CHisIEC**（中国历史铭文语料库）或各大博物馆开放数据集
- 语料规模扩充至500-1000条后，预期术语准确率从当前约 10-30% 提升至 60%+
- 同步扩展 `DH_TERM_SET` 术语表，纳入更多细分领域词汇

**论文价值**：语料构建本身即是一项可独立发表的 DH 贡献。

## 方向二：引入检索增强（RAG）

**当前局限**：LLaVA-Next 和 LLaMA-3 的知识仅来自预训练，无法引用具体文献。

**改进路径**：
- 构建 DH 文献向量数据库（ChromaDB / FAISS + `sentence-transformers` 中文编码器）
- 在术语解释阶段：先检索最相关的文献片段，再将其拼接到提示词前缀
- 输出的术语解释将包含文献引用，可直接用于论文注释

**论文价值**：RAG + DH 领域知识库是目前 NLP × DH 交叉研究的活跃方向。

## 方向三：多视频批处理与元数据目录生成

**当前局限**：管线一次只处理一条视频，输出一份报告。

**改进路径**：
- 将 `research_report` 的生成逻辑封装为函数 `process_video(path) -> dict`
- 对视频档案批量调用，将所有报告汇总为一个结构化元数据目录（CSV 或数据库）
- 在目录上构建全文检索索引，支持按术语、时期、文物类型检索

**论文价值**：元数据自动化生成是数字档案馆的核心技术需求，具有明确的应用场景。

# 第10章  顶会案例分析

## 案例：《CLUE: A Chinese Language Understanding Evaluation Benchmark》
### COLING 2020 / 中文 NLP 评估基准

> COLING（国际计算语言学大会）与 ACL、EMNLP 并列为 NLP 领域三大顶会，2年一届，2020届录用率约26%。

**论文核心**：Xu et al. (2020) 构建了涵盖9项任务的中文语言理解评估基准 CLUE，为中文 NLP 模型提供了统一的评估框架。9个任务覆盖情感分析、语义相似度、阅读理解等场景，每个任务使用独立的最适指标，而非强行统一。

**实验设计逻辑（与本课程最相关的部分）**：
- **多指标并用**：不同任务使用不同指标（准确率、F1、EM），与本课程 L8 同时使用 BLEU 和术语准确率的思路一致——单一指标无法全面反映模型能力
- **基线设计**：论文同时报告人类基线（Human Performance）和多个预训练模型的分数，清楚划定了"可解空间"的上界——这是评估实验必须报告的对照
- **小数据有效性**：即便是顶会论文，部分子任务（如 TNEWS 新闻分类）训练集只有 53,360 条、测试集 10,000 条——说明任务定义的清晰度比数据规模更重要
- **局限诚实性**：论文明确指出模型在阅读理解任务上与人类差距最大，并分析原因是中文长文本推理能力不足——这种诚实的限制分析是顶会论文的标志

> 参考文献：Xu, L., et al. (2020). CLUE: A Chinese Language Understanding Evaluation Benchmark. *In Proceedings of the 28th International Conference on Computational Linguistics (COLING 2020)*.

## 案例启示：如何设计一个可发表的 DH+LLM 评估实验

| CLUE 的做法 | 对应本课程的实践 |
|------------|----------------|
| 为每个任务单独选择最适指标 | L8：摘要用 BLEU，术语识别用准确率——任务不同，指标不同 |
| 同时报告人类基线 | 论文中加入人工评估小样本，与模型分数对比，划定上界 |
| 透明说明数据构建方法 | L3 数据说明：合成10条，人工核验，来源完整引用 |
| 对比不同规模/配置的模型 | 本课程可比较：基础 LLaMA-3 vs QLoRA 微调版 |
| 明确指出模型失败的任务类型 | L8 低分样本分析：「数字孪生」等低频术语准确率接近0 |

**核心教训**：评估实验的可信度来自**任务定义的清晰度**和**局限说明的诚实度**，不来自指标数值的高低。一篇10条数据、诚实报告边界的论文，比50条数据、过度解读指标的论文更有学术价值。

# 第11章  论文选题指南

## 如何确定一个 DH + LLM 研究选题

- **从语料缺口出发，而非从模型出发**：问「哪类文化遗产材料缺乏可计算的分析工具？」，而不是「我想用 GPT-4 做什么？」

- **锁定一种具体的文化材料或档案挑战**：甲骨文字形识别、少数民族非遗口述史转写、明清地方志实体抽取——每一个都是具体且可测量的问题

- **在选题阶段就定义评估指标**：「我怎么知道我的系统比现有方法更好？」如果这个问题没有答案，选题就尚未成熟

## 小规模实验设计原则

- **10-50 条精标注样本足以做概念验证**：研究生论文不需要 ImageNet 级别的数据集。10条高质量样本 + 诚实的局限说明，比1000条噪声数据 + 虚高的指标更有学术价值

- **「一个模型 + 一个任务 + 一个指标」是完整的论文贡献**：本课程的管线包含五个组件——这是研究原型的复杂度，单篇论文应该聚焦其中一个组件的深度分析

- **诚实地报告局限**：指标偏低不是失败，是对领域现状的准确描述。说「当前最优模型在此任务上仍有显著局限」比说「我们达到了 95% 准确率」更难造假、更有价值

# 快速回顾（Quick Check）

- **原型做了什么**：一段数字人文视频 → Whisper 语音转写（L6） → LLaMA-3 文本摘要（L1）+ QLoRA 术语解释（L3） → LLaVA-Next 视觉分析——文字识别/文化元素/档案描述（L7） → BLEU + 术语命中率量化评估（L8） → 论文级柱状图（L9） → 结构化 JSON 研究报告

- **每个阶段的来源**：管线中没有任何一行新模型代码——所有模型均来自 L1/L3/L6/L7；所有评估函数来自 L8；所有图表逻辑来自 L9。本讲只做了一件事：将九讲的独立组件串成一条可运行的研究管线

- **评估说明了什么**：BLEU（0–1 归一化）衡量摘要与原文的文本重叠度；术语命中率（0–1 归一化）衡量领域术语覆盖度。这是单条样本的代理评估，展示了管线可运行性；正式评估需要 50+ 条人工标注参考答案、独立测试集和基线对比（参见 L8）

- **走向发表的下一步**：选择一个组件深化——建议路径：扩充 L3 语料（500 条 → 重训 → L8 系统评估 → 与基础 LLaMA-3 对比）+ 诚实报告局限 = 一篇研究生水平 DH + LLM 论文的完整框架；或者走 RAG 路线（接向量数据库，术语解释带文献引用）

- **最终交付物**：`L10_research_report.json`——包含视频来源、转写全文、摘要、术语解释、LLaVA-Next 三项视觉分析、0–1 归一化评估分数和 ISO 格式时间戳，可直接存档或载入数据库进行批量分析